In [23]:
import os
import time
from datetime import datetime, timedelta

import requests
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm
from dotenv import load_dotenv

from tensorflow.keras.models import Sequential
from transformers import pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.layers import Conv1D, Dense, GlobalMaxPooling1D, Dropout, BatchNormalization

import matplotlib.pyplot as plt

In [24]:
load_dotenv(override=True)

NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")
if not NEWSAPI_KEY:
    raise EnvironmentError("Please set NEWSAPI_KEY environment variable (or in a .env file).")
else:
    print("NEWSAPI key found")

NEWSAPI key found


In [25]:
TICKER = "GLD"
DAYS = 365
QUERY = '("gold price" OR gold OR GLD)'
WINDOW_SIZE = 7

In [26]:
def fetch_news_titles(date_str, api_key):
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": QUERY,
        "from": date_str,
        "to": date_str,
        "language": "en",
        "pageSize": 100,
        "sortBy": "relevancy",
        "apiKey": api_key
    }
    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()
        return [a["title"] for a in data.get("articles", []) if a.get("title")]
    except Exception as e:
        print(f"Error {date_str}: {e}")
        return []

In [27]:
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis",
    return_all_scores=True
)

def compute_daily_sentiment(titles):
    if not titles:
        return {"pos": 0.0, "neg": 0.0, "neu": 0.0, "n": 0}
    
    res = sentiment_pipe(titles)
    pos, neg, neu = [], [], []
    for entry in res:
        scores = {d["label"].lower(): d["score"] for d in entry}
        pos.append(scores.get("positive", 0))
        neg.append(scores.get("negative", 0))
        neu.append(scores.get("neutral", 0))
    
    return {
        "pos": np.mean(pos),
        "neg": np.mean(neg),
        "neu": np.mean(neu),
        "n": len(titles)
    }

Device set to use cpu
c:\Users\User\dev\training_model\.venv\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [28]:
end_date = datetime.utcnow().date() - timedelta(days=1)
start_date = end_date - timedelta(days=DAYS - 1)

daily = []
for i in tqdm(range(DAYS)):
    d = start_date + timedelta(days=i)
    date_str = d.strftime("%Y-%m-%d")
    titles = fetch_news_titles(date_str, NEWSAPI_KEY)
    scores = compute_daily_sentiment(titles)
    scores["date"] = d
    daily.append(scores)

sent_df = pd.DataFrame(daily)
sent_df.head()

C:\Users\User\AppData\Local\Temp\ipykernel_12896\2091947661.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date() - timedelta(days=1)
  1%|          | 2/365 [00:00<01:05,  5.55it/s]

Error 2024-09-10: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2024-09-10&to=2024-09-10&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1
Error 2024-09-11: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2024-09-11&to=2024-09-11&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1


  1%|          | 4/365 [00:00<01:06,  5.47it/s]

Error 2024-09-12: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2024-09-12&to=2024-09-12&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1
Error 2024-09-13: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2024-09-13&to=2024-09-13&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1


  2%|▏         | 6/365 [00:01<01:04,  5.61it/s]

...

100%|█████████▉| 364/365 [01:06<00:00,  5.64it/s]

Error 2025-09-07: 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2025-09-07&to=2025-09-07&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1
Error 2025-09-08: 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2025-09-08&to=2025-09-08&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1


100%|██████████| 365/365 [01:06<00:00,  5.48it/s]

Error 2025-09-09: 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=%28%22gold+price%22+OR+gold+OR+GLD%29&from=2025-09-09&to=2025-09-09&language=en&pageSize=100&sortBy=relevancy&apiKey=257823ea74ad47f09be68a3cf27e8bd1


,pos,neg,neu,n,date
0,0.0,0.0,0.0,0,2024-09-10
1,0.0,0.0,0.0,0,2024-09-11
2,0.0,0.0,0.0,0,2024-09-12
3,0.0,0.0,0.0,0,2024-09-13
4,0.0,0.0,0.0,0,2024-09-14
